In [ ]:
## python modules used within this notebook
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
import matplotlib.colors as colors
import os
import sys
import mynumerics as mn
import units
import HHG
from IPython.display import display, Markdown
from IPython.display import HTML


matplotlib.rcParams['animation.embed_limit'] = 200.
%matplotlib inline

## TDSE with a custom input

We show the interface for the TDSE solver accessed directly through Python. We use this solver for a custom field we define, and then analyse the result in details. We will show the spectrum of the source term, wavefunction, we do energetic analyses via the Gabor transform and [invariant energetic distribution](https://doi.org/10.1103/PhysRevA.106.053115). Finally, we will show the depletion of the ground state.


First, we import the compiled dynamical library and its Pythonic wrapper:

In [ ]:
from PythonTDSE import *

# Compiled dynamic C library
path_to_DLL = os.path.join(os.environ['TDSE_1D_BUILD'],'libsingleTDSE.so')
DLL = TDSE_DLL(path_to_DLL)

### Define the custom input field & numerical parameters

Here we define the input parameters for the CTDSE solver and the initial pulse. We show an example of a chirped pulse with a $\sin^2$-envelope. The field is then given by

$$ \mathcal{E}(t) = \mathcal{E}_0 \sin^2 \left( \frac{t}{T_{\text{envelope}}} \right) \cos \left(\omega_0 t + \omega_c t^2 \right) \,.$$

(Note that the instantaneous frequency is then $\omega_i(t) = \omega_0 + 2\omega_c t$. This means that $\omega_0$ cannot be taken as the central frequency, the frequency at the peak of the pulse is $\omega_i(\pi T_{\text{envelope}}/2) = \omega_0 + \pi \omega_c T_{\text{envelope}}$.)

In [ ]:
omega0 = mn.ConvertPhoton(800e-9,'lambdaSI','omegaau')
chirp = 0*2e-4
E_0 = 0.15   # peak electric field amplitude

T0 = mn.ConvertPhoton(omega0,'omegaau','T0au') # the duration of the reference cycle
T_max = 7*T0 # total pulse duration expressed in the number of the reference cycles
N_t = 12000  # # of points for field construction (not for TDSE)

# Construct the field
tgrid = np.linspace(0, T_max, N_t)
E = E_0* (np.sin(np.pi*tgrid/T_max)**2) *np.cos(omega0*tgrid + chirp*(tgrid)**2)


# Create instance of input structure
inputs = inputs_def()

# Set the inputs for the TDSE solver
trg_a = HHG.soft_Coulomb_a['Ar'] # soft-Coulomb-potnetial parameter from HHG module 
inputs.init_default_inputs(
            Eguess   = -HHG.Ip_list['Ar'] ,       # ionisation potential from the HHG module 
            trg_a    = trg_a, 
            dt       = 0.125 ,
            dx       = 0.4 ,
            num_r    = 16000 ,
            writewft = 1 ,
            tprint   = 1. ,
            x_int    = 2. )
# Note: Parameters currently needs to be fixed for the gauge-invariant energetic analysis (gas & some of numerics for the same ensemble of bound states)

The parameters are fixed to match [the energetic analysis](#invariant_energy_distribution).

### Pipeline to execute the TDSE computation

In [ ]:
inputs.init_time_and_field(DLL, E = E, t = tgrid) # set our electric field as the input
DLL.init_GS(inputs)                               # create the C-types input for the C-library
output = outputs_def()                            # prepare the structure that holds the TDSE outputs 
DLL.call1DTDSE(inputs, output)                    # run TDSE

### Obtain detailed analyses and visualisation
Here we specify some parameters for various analyses and plotting

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


tgrid = output.get_tgrid()

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    tgrid,
    output.get_Efield() / np.max(np.abs(output.get_Efield())),
    label='Electric field'
)

ax.plot(
    tgrid,
    output.get_sourceterm() / np.max(np.abs(output.get_sourceterm())),
    label='Source term'
)

ax.plot(
    tgrid,
    output.get_expval() / np.max(np.abs(output.get_expval())),
    label='Expectation value'
)

ax.set_xlim(tgrid[[0, -1]])
ax.set_ylim(-1, 1)
ax.set_xlabel(r'$t~[\mathrm{a.u.}]$')
ax.set_ylabel('Normalised value')
ax.legend()

fig.tight_layout()
plt.show()